In [3]:
public enum OrderStatus
{
    Created,
    InProgress,
    Completed
}

public class Customer
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string PhoneNumber { get; set; }
    
    public Customer(string firstName, string lastName, string phoneNumber)
    {
        FirstName = firstName;
        LastName = lastName;
        PhoneNumber = phoneNumber;
    }
    
    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }
    
    public override string ToString()
    {
        return $"{GetFullName()} ({PhoneNumber})";
    }
}

public class Employee
{
    public string FirstName { get; set; }
    public string LastName { get; set; }
    public string Position { get; set; }
    public int OrdersProcessed { get; set; }
    
    public Employee(string firstName, string lastName, string position)
    {
        FirstName = firstName;
        LastName = lastName;
        Position = position;
        OrdersProcessed = 0;
    }
    
    public string GetFullName()
    {
        return $"{FirstName} {LastName}";
    }
    
    public void IncrementOrdersProcessed()
    {
        OrdersProcessed++;
    }
    
    public override string ToString()
    {
        return $"{GetFullName()} - {Position} (Обработано заказов: {OrdersProcessed})";
    }
}

public class Order
{
    private static int _nextOrderId = 1;
    
    public int OrderId { get; private set; }
    public string Description { get; set; }
    public Employee AssignedEmployee { get; set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; private set; }
    public Customer Customer { get; set; }
    
    public event Action<Order, OrderStatus> StatusChanged;
    
    public Order(string description, Customer customer)
    {
        OrderId = _nextOrderId++;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }
    
    public void ChangeStatus(OrderStatus newStatus)
    {
        var oldStatus = Status;
        Status = newStatus;
        
        if (newStatus == OrderStatus.Completed && AssignedEmployee != null)
        {
            AssignedEmployee.IncrementOrdersProcessed();
        }
        
        StatusChanged?.Invoke(this, oldStatus);
    }
    
    public override string ToString()
    {
        return $"Заказ #{OrderId}: {Description} | Статус: {Status} | Создан: {CreatedAt:dd.MM.yyyy}";
    }
}

public class Company
{
    public string Name { get; set; }
    public List<Employee> Employees { get; private set; }
    public List<Order> Orders { get; private set; }
    public List<Customer> Customers { get; private set; }
    
    public delegate void NotificationHandler(string message);
    public event NotificationHandler OnNotification;
    
    public Company(string name)
    {
        Name = name;
        Employees = new List<Employee>();
        Orders = new List<Order>();
        Customers = new List<Customer>();
    }
    
    public void AddEmployee(Employee employee)
    {
        Employees.Add(employee);
        Notify($"Сотрудник {employee.GetFullName()} добавлен в компанию");
    }
    
    public void RemoveEmployee(Employee employee)
    {
        Employees.Remove(employee);
        Notify($"Сотрудник {employee.GetFullName()} удален из компании");
    }
    
    public Employee FindEmployeeByName(string firstName, string lastName)
    {
        return Employees.FirstOrDefault(e => 
            e.FirstName.Equals(firstName, StringComparison.OrdinalIgnoreCase) && 
            e.LastName.Equals(lastName, StringComparison.OrdinalIgnoreCase));
    }
    
    public void AddCustomer(Customer customer)
    {
        Customers.Add(customer);
        Notify($"Заказчик {customer.GetFullName()} добавлен в систему");
    }
    
    public Customer FindCustomerByName(string firstName, string lastName)
    {
        return Customers.FirstOrDefault(c => 
            c.FirstName.Equals(firstName, StringComparison.OrdinalIgnoreCase) && 
            c.LastName.Equals(lastName, StringComparison.OrdinalIgnoreCase));
    }
    
    public void AddOrder(Order order)
    {
        Orders.Add(order);
        
        order.StatusChanged += OnOrderStatusChanged;
        
        Notify($"Заказ #{order.OrderId} добавлен в систему");
    }
    
    public void AssignEmployeeToOrder(int orderId, Employee employee)
    {
        var order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order != null)
        {
            order.AssignedEmployee = employee;
            Notify($"Сотрудник {employee.GetFullName()} назначен на заказ #{orderId}");
        }
    }
    
    public void ChangeOrderStatus(int orderId, OrderStatus newStatus)
    {
        var order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order != null)
        {
            order.ChangeStatus(newStatus);
        }
    }
    
    public List<Order> GetOrdersByStatus(OrderStatus status)
    {
        return Orders.Where(o => o.Status == status).ToList();
    }
    
    public List<Order> GetOrdersByCustomer(string firstName, string lastName)
    {
        return Orders.Where(o => 
            o.Customer.FirstName.Equals(firstName, StringComparison.OrdinalIgnoreCase) && 
            o.Customer.LastName.Equals(lastName, StringComparison.OrdinalIgnoreCase)).ToList();
    }
    
    public void GenerateEmployeeReport()
    {
        Notify("=== ОТЧЕТ ПО СОТРУДНИКАМ ===");
        foreach (var employee in Employees.OrderByDescending(e => e.OrdersProcessed))
        {
            Notify($"{employee.GetFullName()} - {employee.Position}: {employee.OrdersProcessed} заказов");
        }
        Notify("=== КОНЕЦ ОТЧЕТА ===");
    }
    
    private void OnOrderStatusChanged(Order order, OrderStatus oldStatus)
    {
        string employeeInfo = order.AssignedEmployee != null ? 
            $" (Ответственный: {order.AssignedEmployee.GetFullName()})" : "";
            
        Notify($"Статус заказа #{order.OrderId} изменен: {oldStatus} -> {order.Status}{employeeInfo}");
    }
    
    private void Notify(string message)
    {
        OnNotification?.Invoke(message);
    }
}

Company company = new Company("ТехноПрофи");
        
company.OnNotification += message => Console.WriteLine($"[Уведомление] {message}");
        
company.AddEmployee(new Employee("Иван", "Петров", "Менеджер"));
company.AddEmployee(new Employee("Мария", "Сидорова", "Разработчик"));
company.AddEmployee(new Employee("Алексей", "Козлов", "Тестировщик"));
        
company.AddCustomer(new Customer("ООО", "Ромашка", "+7-999-123-45-67"));
company.AddCustomer(new Customer("ИП", "Иванов", "+7-999-765-43-21"));
        
var customer1 = company.FindCustomerByName("ООО", "Ромашка");
var customer2 = company.FindCustomerByName("ИП", "Иванов");
        
var order1 = new Order("Разработка веб-сайта", customer1);
var order2 = new Order("Создание мобильного приложения", customer2);
var order3 = new Order("Тестирование системы", customer1);
        
company.AddOrder(order1);
company.AddOrder(order2);
company.AddOrder(order3);
        
var employee1 = company.FindEmployeeByName("Иван", "Петров");
var employee2 = company.FindEmployeeByName("Мария", "Сидорова");
        
company.AssignEmployeeToOrder(1, employee1);
company.AssignEmployeeToOrder(2, employee2);
company.AssignEmployeeToOrder(3, employee1);
        
company.ChangeOrderStatus(1, OrderStatus.InProgress);
company.ChangeOrderStatus(2, OrderStatus.InProgress);
company.ChangeOrderStatus(1, OrderStatus.Completed);
company.ChangeOrderStatus(3, OrderStatus.InProgress);
company.ChangeOrderStatus(2, OrderStatus.Completed);
        
Console.WriteLine("\n=== ЗАКАЗЫ В РАБОТЕ ===");
var inProgressOrders = company.GetOrdersByStatus(OrderStatus.InProgress);
foreach (var order in inProgressOrders)
{
    Console.WriteLine(order);
}
        
Console.WriteLine("\n=== ЗАКАЗЫ ООО 'РОМАШКА' ===");
var customerOrders = company.GetOrdersByCustomer("ООО", "Ромашка");
foreach (var order in customerOrders)
{
    Console.WriteLine(order);
}
        
Console.WriteLine();
company.GenerateEmployeeReport();

[Уведомление] Сотрудник Иван Петров добавлен в компанию
[Уведомление] Сотрудник Мария Сидорова добавлен в компанию
[Уведомление] Сотрудник Алексей Козлов добавлен в компанию
[Уведомление] Заказчик ООО Ромашка добавлен в систему
[Уведомление] Заказчик ИП Иванов добавлен в систему
[Уведомление] Заказ #1 добавлен в систему
[Уведомление] Заказ #2 добавлен в систему
[Уведомление] Заказ #3 добавлен в систему
[Уведомление] Сотрудник Иван Петров назначен на заказ #1
[Уведомление] Сотрудник Мария Сидорова назначен на заказ #2
[Уведомление] Сотрудник Иван Петров назначен на заказ #3
[Уведомление] Статус заказа #1 изменен: Created -> InProgress (Ответственный: Иван Петров)
[Уведомление] Статус заказа #2 изменен: Created -> InProgress (Ответственный: Мария Сидорова)
[Уведомление] Статус заказа #1 изменен: InProgress -> Completed (Ответственный: Иван Петров)
[Уведомление] Статус заказа #3 изменен: Created -> InProgress (Ответственный: Иван Петров)
[Уведомление] Статус заказа #2 изменен: InProgress 